In [ ]:
import os
import pyarrow.parquet as pq
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
from tqdm import tqdm

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from torch.utils.data import TensorDataset, DataLoader
from torch.serialization import safe_globals

## Train

In [ ]:
# ---------------------------------------
# Configuration
# ---------------------------------------
parquet_path = 'Master_cleaned.parquet'
return_col   = 'ret_5d_future'  # Column to predict
batch_size   = 4096
n_epochs     = 3  # number of training epochs
patience = 3  # number of epochs with no improvement before stopping
device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ---------------------------------------
# Prepare ParquetFile and feature columns
# ---------------------------------------
pf = pq.ParquetFile(parquet_path)
all_cols = pf.schema_arrow.names
exclude  = {return_col, 'date', 'ticker'}
feature_cols = [c for c in all_cols if c not in exclude]

# ---------------------------------------
# Precompute date-wise cutoff map
# ---------------------------------------
cutoff_lists, all_dates = {}, set()
for batch in pf.iter_batches(batch_size=1_500_000):
    dfb = batch.to_pandas()[['date', return_col]]
    for date, grp in dfb.groupby('date'):
        all_dates.add(date)
        cutoff_lists.setdefault(date, []).extend(grp[return_col].values)
cutoff_map = {date: np.quantile(vals, 0.95) for date, vals in cutoff_lists.items()}
all_dates = sorted(all_dates)
split_idx = int(len(all_dates) * 0.8)
train_dates, val_dates = set(all_dates[:split_idx]), set(all_dates[split_idx:])

# ---------------------------------------
# Fit scaler on a sample row group
# ---------------------------------------
sample = pf.read_row_group(0).to_pandas()[feature_cols]
scaler = StandardScaler().fit(sample.values)

# ---------------------------------------
# Define IterableDataset for streaming
# ---------------------------------------
class ParquetDataset(IterableDataset):
    def __init__(self, dates_subset):
        self.dates_subset = dates_subset

    def __iter__(self):
        local_pf = pq.ParquetFile(parquet_path)
        for batch in local_pf.iter_batches(batch_size=batch_size):
            df = batch.to_pandas()
            df = df[df['date'].isin(self.dates_subset)]
            if df.empty:
                continue
            df['label'] = (df[return_col] >= df['date'].map(cutoff_map)).astype(np.int8)
            X_raw = df[feature_cols].values.astype(np.float32)
            y_raw = df['label'].values.astype(np.float32)
            mask = np.isfinite(X_raw).all(axis=1)
            if not mask.any():
                continue
            X = scaler.transform(X_raw[mask])
            y = y_raw[mask]
            for i in range(0, len(X), batch_size):
                xb = torch.from_numpy(X[i:i+batch_size]).to(device)
                yb = torch.from_numpy(y[i:i+batch_size]).to(device)
                yield xb, yb

# ---------------------------------------
# Create DataLoaders without multiprocessing
# ---------------------------------------
train_ds = ParquetDataset(train_dates)
val_ds   = ParquetDataset(val_dates)
train_loader = DataLoader(train_ds, batch_size=None, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=None, num_workers=0)

# Count positives/negatives for weighting
print("Counting positive/negative samples for pos_weight...")
pos_count = 0
neg_count = 0
for batch in pf.iter_batches(batch_size=1_500_000):
    dfb = batch.to_pandas()[['date', return_col]]
    dfb['label'] = (dfb[return_col] >= dfb['date'].map(cutoff_map)).astype(np.int8)
    pos_count += (dfb['label'] == 1).sum()
    neg_count += (dfb['label'] == 0).sum()

pos_weight_value = neg_count / max(pos_count, 1)  # avoid division by zero
print(f"pos_weight = {pos_weight_value:.2f}")
pos_weight = torch.tensor(pos_weight_value, dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ---------------------------------------
# Define the MLP model
# ---------------------------------------
class SelectorMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = SelectorMLP(len(feature_cols)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Estimate steps per epoch
total_rows   = sum(pf.metadata.row_group(i).num_rows for i in range(pf.num_row_groups))
train_steps  = int(total_rows * 0.8) // batch_size
val_steps    = int(total_rows * 0.2) // batch_size

# ---------------------------------------
# Training loop with tqdm progress bars
# ---------------------------------------
best_val_auc, epochs_no_improve = 0.0, 0

for epoch in range(1, n_epochs + 1):
    # Training
    model.train()
    train_loss = 0.0
    pbar_train = tqdm(train_loader, total=train_steps, desc=f"Epoch {epoch}/{n_epochs} Train", dynamic_ncols=True)
    for batch_idx, (X_batch, y_batch) in enumerate(pbar_train, 1):
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pbar_train.set_postfix(loss=train_loss / batch_idx)
        
    avg_train_loss = train_loss / train_steps

    # Validation
    model.eval()
    val_loss = 0.0
    y_trues, y_scores = [], []
    pbar_val = tqdm(val_loader, total=val_steps, desc=f"Epoch {epoch}/{n_epochs} Val", dynamic_ncols=True)
    with torch.no_grad():
        for batch_idx, (X_batch, y_batch) in enumerate(pbar_val, 1):
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item()
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            y_scores.append(probs)
            y_trues.append(y_batch.cpu().numpy())
            pbar_val.set_postfix(loss=val_loss / batch_idx)
            
    avg_val_loss = val_loss / val_steps
    y_trues = np.concatenate(y_trues)
    y_scores = np.concatenate(y_scores)
    val_auc = roc_auc_score(y_trues, y_scores)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'selector_mlp_state_dict.pt')
        torch.jit.script(model).save('selector_mlp_scripted.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break

    print(f"Epoch {epoch}/{n_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}")


print("Training complete. Saved:")
print(" - Best state dict -> best_selector_state_dict.pt")
print(" - Best TorchScript model -> best_selector_scripted.pt")


## Test

In [ ]:
# ---------------------------------------
# Configuration
# ---------------------------------------
parquet_path   = 'Master_cleaned.parquet'
return_col     = 'ret_5d_future'
batch_size     = 1_500_000   # streaming size for Parquet
test_fraction  = 0.2         # last 20% of dates for test
device         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# ---------------------------------------
# Build X_test, y_test (same as before)
# ---------------------------------------
pf = pq.ParquetFile(parquet_path)
all_cols = pf.schema_arrow.names
exclude  = {return_col, 'date', 'ticker'}
feature_cols = [c for c in all_cols if c not in exclude]

cutoff_lists, all_dates = {}, set()
for batch in pf.iter_batches(batch_size=batch_size):
    dfb = batch.to_pandas()[['date', return_col]]
    for date, grp in dfb.groupby('date'):
        all_dates.add(date)
        cutoff_lists.setdefault(date, []).extend(grp[return_col].values)
cutoff_map = {d: np.quantile(vals, 0.95) for d, vals in cutoff_lists.items()}
all_dates = sorted(all_dates)

split_idx  = int(len(all_dates) * (1 - test_fraction))
test_dates = set(all_dates[split_idx:])
# split_idx = next(i for i, d in enumerate(all_dates) if str(d) >= '2025-06-15')
# test_dates = set(all_dates[split_idx:])

sample = pf.read_row_group(0).to_pandas()[feature_cols]
scaler = StandardScaler().fit(sample.values)

chunks = []
for batch in pf.iter_batches(batch_size=batch_size):
    df = batch.to_pandas()
    df = df[df['date'].isin(test_dates)]
    if not df.empty:
        chunks.append(df)
test_df = pd.concat(chunks, ignore_index=True)

test_df['label'] = (test_df[return_col] >= test_df['date'].map(cutoff_map)).astype(np.int8)
mask = np.isfinite(test_df[feature_cols].values).all(axis=1)
test_df = test_df.loc[mask].reset_index(drop=True)

X_test = scaler.transform(test_df[feature_cols].values.astype(np.float32))
y_test = test_df['label'].values.astype(np.int8)
print("X_test shape:", X_test.shape, "y_test dist:", np.bincount(y_test))

# ---------------------------------------
# Load the TorchScript model
# ---------------------------------------
scripted_model = torch.jit.load('selector_mlp_scripted.pt', map_location=device)
scripted_model.to(device).eval()

# ---------------------------------------
# Batched inference & metrics
# ---------------------------------------
test_ds     = TensorDataset(torch.from_numpy(X_test).float(),
                            torch.from_numpy(y_test).long())
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False, num_workers=0)

all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        logits = scripted_model(Xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(np.int8)

        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(yb.numpy())

all_probs  = np.concatenate(all_probs)
all_preds  = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

auc      = roc_auc_score(all_labels, all_probs)
accuracy = accuracy_score(all_labels, all_preds)

print(f"\nTest AUC:      {auc:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n",
      classification_report(all_labels, all_preds, digits=4))

test_df["prob"] = all_probs
test_df["pred"] = all_preds  # Optional: for inspection
test_df["rank_prob"] = test_df.groupby("date")["prob"].rank(method="first", ascending=False)
top_150_df = test_df[test_df["rank_prob"] <= 150].copy()
top_150_df = top_150_df.sort_values(["date", "rank_prob"])
top_150_df[["date", "ticker", "prob", "rank_prob", "ret_5d_future", "open"]].to_csv("top_150_predictions.csv", index=False)
print("Exported top 150 predictions per date to top_150_predictions.csv")